# cpet.rag — Phase 1 실인입 (Colab)
학술 PDF를 **실제로** 파싱(Docling)→Late Chunking 임베딩(Jina-v3 API)→LanceDB 적재까지 돌립니다.

**준비물**
1. 런타임: GPU 권장 (Runtime → Change runtime type → T4). Jina는 API라 GPU 없이도 동작하지만 Docling이 빨라집니다.
2. Colab Secrets(🔑 좌측 키 아이콘)에 등록: `JINA_API_KEY`(필수), `GEMINI_API_KEY`(선택, 복잡 표/수식 VLM 폴백), `GH_TOKEN`(private repo 클론용 GitHub PAT)
3. Google Drive에 코퍼스 PDF (`sinico.papers/pdf/`)


## 1. GPU 확인

In [ ]:
!nvidia-smi || echo 'GPU 없음 — CPU로도 진행 가능(느림)'

## 2. Repo 클론
public repo라 토큰 불필요. (private로 두려면 Colab Secrets에 `GH_TOKEN` 등록 — 없어도 아래 코드는 그냥 진행)

In [ ]:
import os, subprocess
def secret(k):
    try:
        from google.colab import userdata; return userdata.get(k)
    except Exception:
        return None

%cd /content
GH_TOKEN = secret('GH_TOKEN')
REPO = 'github.com/cyanluna-git/cpet.rag.git'
DEST = '/content/cpet.rag'   # 절대경로 — 중첩 방지
if os.path.isdir(DEST + '/.git'):
    subprocess.run(['git','-C',DEST,'pull','--ff-only'], check=False)
else:
    url = f'https://{GH_TOKEN}@{REPO}' if GH_TOKEN else f'https://{REPO}'
    subprocess.run(['git','clone',url,DEST], check=True)
%cd /content/cpet.rag
!git log --oneline -1

## 3. 의존성 설치
docling·torch·lancedb 등 — 수 분 소요(최초).

In [ ]:
!pip install -q -e '.[ingestion,vectorstore,gpu]'
print('설치 완료')

## 4. Drive 마운트 + PDF 경로

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
PDF_DIR = '/content/drive/MyDrive/06_학습_기술/FZCare 참여연구원/sinico.papers/pdf'
import os; print('PDF 개수:', len([f for f in os.listdir(PDF_DIR) if f.endswith('.pdf')]) if os.path.isdir(PDF_DIR) else 'PDF_DIR 경로 확인 필요')

## 5. 키·설정
Jina API + 로컬 LanceDB(PoC). 운영은 S3(Phase 4).

In [ ]:
import os
# secret()는 위 클론 셀에서 정의됨 (없으면 아래 재정의)
try: secret
except NameError:
    def secret(k):
        try:
            from google.colab import userdata; return userdata.get(k)
        except Exception: return None

os.environ['JINA_API_KEY']   = secret('JINA_API_KEY') or ''
os.environ['GEMINI_API_KEY'] = secret('GEMINI_API_KEY') or ''   # 선택
os.environ['EMBED_MODEL']    = 'jina-embeddings-v3'
os.environ['EMBED_DIM']      = '1024'
os.environ['LANCEDB_URI']    = '/content/cpet_lancedb'   # PoC 로컬 (운영은 s3://)
assert os.environ['JINA_API_KEY'], '❌ JINA_API_KEY Secret을 등록하세요 (좌측 🔑)'
from core.config import settings
print('embed', settings.embed_model, settings.embed_dim, '| VLM 폴백', bool(os.environ['GEMINI_API_KEY']))

## 6. 스모크 — 코퍼스 메타 로드

In [ ]:
from core.metadata import load_corpus_index
papers = load_corpus_index('data/corpus_index.csv')
have = [p for p in papers if p.file and os.path.exists(os.path.join(PDF_DIR, p.file))]
print('메타', len(papers), '| PDF 존재', len(have))

## 7. 샘플 실인입 (10편)
실제 Docling 파싱 + Jina-v3 Late Chunking + LanceDB 적재. 증분이라 재실행 시 skip.

In [ ]:
from ingestion import ingest_corpus
from ingestion.embed import JinaEmbedder
from core.vectorstore import LanceDBStore
from ingestion.load import ProcessedRegistry

embedder = JinaEmbedder(backend='api', model=settings.embed_model, dim=settings.embed_dim)
store    = LanceDBStore(uri=settings.lancedb_uri, dim=settings.embed_dim)
registry = ProcessedRegistry()

sample = have[:10]
results = ingest_corpus(sample, PDF_DIR, store=store, embedder=embedder,
                        registry=registry, use_vlm=bool(os.environ['GEMINI_API_KEY']))
from collections import Counter
print('결과:', dict(Counter(r.status for r in results)))
print('적재 청크:', store.count())

## 8. 검증 — 실제 쿼리 검색
임베딩된 청크에 벡터+FTS 검색이 되는지 확인 (Phase 2에서 번역·리랭크·생성으로 고도화).

In [ ]:
# 벡터 검색 (쿼리도 Jina로 임베딩)
qv = embedder.embed(['skeletal muscle energy metabolism during exercise'])[0]
for r in store.search(qv, top_k=3):
    print(round(r.score,3), '|', (r.chunk.doi or '')[:30], '|', r.chunk.text[:80].replace(chr(10),' '))
print('--- FTS ---')
for r in store.fts('lactate', top_k=3):
    print('|', (r.chunk.doi or '')[:30], '|', r.chunk.text[:80].replace(chr(10),' '))

## 9. (선택) Obsidian Vault 생성

In [ ]:
from ingestion.obsidian import write_vault
n = write_vault(sample, '/content/cpet_vault')   # fetch_citations=True 면 인용그래프(네트워크)
print('노트', n, '개 → /content/cpet_vault')

## 전체 794편 인입 (시간 소요)

**증분 실행** — registry에 기록된 논문은 자동 skip.
중단 후 재실행해도 완료된 논문부터 이어 진행됩니다.

예상 소요: GPU(T4) 기준 논문당 약 30-90초 (Docling 파싱 지배). 전체 약 6-20시간.

In [ ]:
# 전체 794편 인입 — have/store/embedder/registry 는 위 샘플 셀에서 정의됨
# 증분이라 중단 후 재실행 가능 (완료 논문은 skipped 처리)
from ingestion import ingest_corpus
import os

results = ingest_corpus(
    have, PDF_DIR,
    store=store,
    embedder=embedder,
    registry=registry,
    use_vlm=bool(os.environ.get('GEMINI_API_KEY')),
)

from collections import Counter
tally = dict(Counter(r.status for r in results))
total_chunks = sum(r.n_chunks for r in results)
print('결과:', tally)
print('총 청크:', total_chunks)
print('store.count():', store.count())

# 실패 목록 확인
failed = [r for r in results if r.status == 'failed']
if failed:
    print('\n실패 목록:')
    for r in failed:
        print(f'  {r.doi or r.paper_key}: {r.error}')

## 10. 다음 단계
- 샘플이 잘 되면 **전체 794편**: `ingest_corpus(have, PDF_DIR, ...)` (limit 제거) — 카드 **#3122**.
- LanceDB를 Drive나 S3에 영속화 (운영은 Phase 4 S3).
- **Phase 2**(#3121~#3126): Query Translation·하이브리드·Reranker·Strict Citation 생성.

> Docling을 GPU로 돌리려면 파이프라인에 device='cuda' 전달이 필요(현재 기본 CPU). 샘플은 CPU로 충분, 전체 배치 전 소폭 보강 예정.